# RacCSB20637 and relatives

Full analysis for mapping GARD breakpoints to final complete alignment `RacCSB20637_relatives_GARD_020924_linsi_trim.fas`


In [8]:
from Bio import SeqIO

In [9]:
def thelens(thel):
    for i in range(len(thel)-1):
        print(thel[i+1] - thel[i])
        
def rm_shorts(thel, thresh):
    newl = [thel[0]]
    for i in range(len(thel)-1):
        if thel[i+1] - thel[i] > thresh:
            newl.append(thel[i+1])
    return newl



def mapals(posl, alone, altwo):
    
    #choose reference seq
    records = SeqIO.to_dict(SeqIO.parse(alone, 'fasta'))
    numgaps = [str(records[n].seq).count('-') for n in list(records)]
    ref = list(records)[numgaps.index(min(numgaps))]
    refseqone = str(records[ref].seq)
    
    rawpos = []        
    for p in posl:
        seqseg = refseqone[:p]
        seqseg = seqseg.replace('-', '')
        rawpos.append(len(seqseg))
        
    rectwo = SeqIO.to_dict(SeqIO.parse(altwo, 'fasta'))
    refseqtwo = str(rectwo[ref].seq)
    
    themap = {}
    c = 0
    for i in range(len(refseqtwo)):
        if str(c) not in list(themap):
            themap.update({str(c):[i]})
        else:
            themap[str(c)].append(i)
        if refseqtwo[i] != '-':
            c = c +1
    
    finalpos = [max(themap[str(x)]) for x in rawpos]
    
    return(finalpos)
    
    

In [10]:
#raw breakpoints
sc2_bps2 = [557,1780,2706,3915,4530,5058,5750,6459,7536,9933,11106,13625,14192,15071,16904,17942,18986,19514,20594,21296,21419,22330,22449,23382,24297,25164,25568,25916,27313,27774,28176]


The above breakpoint positions outputted from GARD are based on a reduced alignment with trimmed ends 

First readjust to non-trimmed coordinates by adding 229nt to each position:

In [11]:
sc2bps_notrim = {'bp2.0': [x + 229 for x in sc2_bps2]}

Now map all coordinates to the current full alignment 

In [12]:
alone = 'RacCSB20637_relatives_GARD_020924_linsi.fas'
altwo = '../alignments/nov23update_250vir_SC2like_RacCS20_mafft.fas'
sc2bps = {x:mapals(sc2bps_notrim[x], alone, altwo) for x in list(sc2bps_notrim)}

print(sc2bps)

{'bp2.0': [830, 2053, 2982, 4318, 4933, 5464, 6156, 6868, 7945, 10342, 11515, 14034, 14601, 15480, 17313, 18351, 19395, 19923, 21003, 21705, 21828, 22816, 22935, 23868, 24803, 25670, 26075, 26423, 27852, 28319, 28772]}


Finally, merge breakpoints less than 100 positions apart (as in original protocol)

In [8]:
sc2bps_fn = {x:rm_shorts(sc2bps[x], 100) for x in list(sc2bps)}

print(sc2bps_fn)

{'bp2.0': [830, 2053, 2982, 4318, 4933, 5464, 6156, 6868, 7945, 10342, 11515, 14034, 14601, 15480, 17313, 18351, 19395, 19923, 21003, 21705, 21828, 22816, 22935, 23868, 24803, 25670, 26075, 26423, 27852, 28319, 28772]}


GARD breakpoints for stages/2 on `nov23update_250vir_SC2like_RacCS20_mafft.fas` -> 

---
[830, 2053, 2982, 4318, 4933, 5464, 6156, 6868, 7945, 10342, 11515, 14034, 14601, 15480, 17313, 18351, 19395, 19923, 21003, 21705, 21828, 22816, 22935, 23868, 24803, 25670, 26075, 26423, 27852, 28319, 28772]

---

In [8]:
len(sc2bps_fn['bp2.0'])

31

<br>
<br>


## Map breakpoints on RacCS20637

In [15]:
alndic = SeqIO.to_dict(SeqIO.parse('../alignments/nov23update_250vir_SC2like_RacCS20_mafft.fas', 'fasta'))
raccs20637_seq = str(alndic['RacCS20637'].seq)
   
fnalnbps = [830, 2053, 2982, 4318, 4933, 5464, 6156, 6868, 7945, 10342, 11515, 14034, 14601, 15480, 17313, 18351, 19395, 19923, 21003, 21705, 21828, 22816, 22935, 23868, 24803, 25670, 26075, 26423, 27852, 28319, 28772]

rawpos = []        
for p in fnalnbps:
    seqseg = raccs20637_seq[:p]
    seqseg = seqseg.replace('-', '')
    rawpos.append(len(seqseg))
    
print('GARD positions on RacCS20637 genome:')
print(rawpos)

GARD positions on RacCS20637 genome:
[677, 1900, 2826, 3984, 4599, 5127, 5819, 6528, 7605, 10002, 11175, 13694, 14261, 15140, 16973, 18011, 19055, 19583, 20663, 21365, 21488, 22368, 22487, 23420, 24335, 25202, 25606, 25954, 27348, 27809, 28198]


In [17]:
len(raccs20637_seq.replace('-', ''))

29805

<br>
<br>

## Relative ORF positions

In [6]:
sc2ref = SeqIO.to_dict(SeqIO.parse('Wuhan-Hu-1.gb', 'gb'))['NC_045512.2']

In [7]:
sc2ref

SeqRecord(seq=Seq('ATTAAAGGTTTATACCTTCCCAGGTAACAAACCAACCAACTTTCGATCTCTTGT...AAA'), id='NC_045512.2', name='NC_045512', description='Severe acute respiratory syndrome coronavirus 2 isolate Wuhan-Hu-1, complete genome', dbxrefs=['BioProject:PRJNA485481'])

In [71]:
allfeats_sc2 = {}

for f in sc2ref.features:

    if f.type == 'CDS':
        if f.qualifiers['product'][0][:3] == 'ORF':
            allfeats_sc2.update({f.qualifiers['product'][0].split(' ')[0]:[int(f.location.start), int(f.location.end)]})
        else:
            allfeats_sc2.update({f.qualifiers['product'][0][0].upper():[int(f.location.start), int(f.location.end)]})
            
    elif f.type == 'mat_peptide':
        allfeats_sc2.update({f.qualifiers['product'][0]:[int(f.location.start), int(f.location.end)]})
        
    elif f.type == "5'UTR":
        allfeats_sc2.update({f.type:[int(f.location.start), int(f.location.end)]})

    elif f.type == "3'UTR":
        allfeats_sc2.update({f.type:[int(f.location.start), int(f.location.end-1)]})        
        
#manually add extra
allfeats_sc2.update({'ORF1b':[13467,21555]})
allfeats_sc2.update({'ORF3b':[25813,25882]})
allfeats_sc2.update({'ORF9b':[28283,28577]})
allfeats_sc2

{"5'UTR": [0, 265],
 'ORF1ab': [265, 21555],
 'leader protein': [265, 805],
 'nsp2': [805, 2719],
 'nsp3': [2719, 8554],
 'nsp4': [8554, 10054],
 '3C-like proteinase': [10054, 10972],
 'nsp6': [10972, 11842],
 'nsp7': [11842, 12091],
 'nsp8': [12091, 12685],
 'nsp9': [12685, 13024],
 'nsp10': [13024, 13441],
 'RNA-dependent RNA polymerase': [13441, 16236],
 'helicase': [16236, 18039],
 "3'-to-5' exonuclease": [18039, 19620],
 'endoRNAse': [19620, 20658],
 "2'-O-ribose methyltransferase": [20658, 21552],
 'ORF1a': [265, 13483],
 'nsp11': [13441, 13480],
 'S': [21562, 25384],
 'ORF3a': [25392, 26220],
 'E': [26244, 26472],
 'M': [26522, 27191],
 'ORF6': [27201, 27387],
 'ORF7a': [27393, 27759],
 'ORF7b': [27755, 27887],
 'ORF8': [27893, 28259],
 'N': [28273, 29533],
 'ORF10': [29557, 29674],
 "3'UTR": [29674, 29902],
 'ORF1b': [13467, 21555],
 'ORF3b': [25813, 25882],
 'ORF9b': [28283, 28577]}

In [72]:
pairal = SeqIO.to_dict(SeqIO.parse('../RacCSB20637_SC2_pair.fas', 'fasta'))

sc2toraccsmap = {}

refc = 0
c= 0

for x,y in zip(str(pairal['MN908947.3'].seq), str(pairal['RacCSB20637.RSL'].seq)):
    if (x != '-') and (y != '-'):
        sc2toraccsmap.update({refc:c})
        refc = refc + 1
        c = c + 1
    elif (x != '-') and (y == '-'):
        sc2toraccsmap.update({refc:c})
        refc = refc +1
    elif (x == '-') and (y != '-'):
        c = c +1
            
sc2toraccsmap[21790]

21750

In [73]:
allfeats_raccs = {x:[sc2toraccsmap[y] for y in allfeats_sc2[x]] for x in list(allfeats_sc2)}
allfeats_raccs

{"5'UTR": [0, 265],
 'ORF1ab': [265, 21516],
 'leader protein': [265, 805],
 'nsp2': [805, 2719],
 'nsp3': [2719, 8515],
 'nsp4': [8515, 10015],
 '3C-like proteinase': [10015, 10933],
 'nsp6': [10933, 11803],
 'nsp7': [11803, 12052],
 'nsp8': [12052, 12646],
 'nsp9': [12646, 12985],
 'nsp10': [12985, 13402],
 'RNA-dependent RNA polymerase': [13402, 16197],
 'helicase': [16197, 18000],
 "3'-to-5' exonuclease": [18000, 19581],
 'endoRNAse': [19581, 20619],
 "2'-O-ribose methyltransferase": [20619, 21513],
 'ORF1a': [265, 13444],
 'nsp11': [13402, 13441],
 'S': [21522, 25320],
 'ORF3a': [25328, 26156],
 'E': [26180, 26408],
 'M': [26458, 27124],
 'ORF6': [27134, 27320],
 'ORF7a': [27326, 27692],
 'ORF7b': [27688, 27820],
 'ORF8': [27826, 28192],
 'N': [28206, 29466],
 'ORF10': [29490, 29607],
 "3'UTR": [29607, 29835],
 'ORF1b': [13428, 21516],
 'ORF3b': [25749, 25818],
 'ORF9b': [28216, 28510]}

In [74]:
fas = ''

for x in list(allfeats_raccs):
    fas = fas + '>%s\n%s\n'%('RacCSB20637_'+x, str(pairal['RacCSB20637.RSL'].seq).replace('-', '')[allfeats_raccs[x][0]:allfeats_raccs[x][1]])
    
with open('../RacCSB20637_all_features.fas', 'w') as out:
    out.write(fas[:-1])

In [75]:
len(str(pairal['RacCSB20637.RSL'].seq).replace('-', ''))

29836

## write gff file

In [78]:
gff_head = """
##gff-version 3
#!gff-spec-version 1.21
#!processor NCBI annotwriter
##sequence-region RacCSB20637 1 29836
##species RacCSB20637
RacCSB20637\tGenbank\tregion\t1\t29836\t.\t+\t.\tID=RacCSB20637
""".strip()

gff_all = gff_head

for x in ['ORF1a', 'ORF1b', 'S', 'ORF3a', 'ORF3b', 'E', 'M', 'ORF6', 'ORF7a', 'ORF7b', 'ORF8', 'N', 'ORF9b', 'ORF10']:

    gff_all = gff_all + "\nRacCSB20637\tGenbank\tCDS\t%i\t%i\t.\t+\t.\tID=%s"%(allfeats_raccs[x][0]+1, allfeats_raccs[x][1], x)
        
print(gff_all)

with open('RacCSB20637.gff3', 'w') as out:
    out.write(gff_all)

##gff-version 3
#!gff-spec-version 1.21
#!processor NCBI annotwriter
##sequence-region RacCSB20637 1 29836
##species RacCSB20637
RacCSB20637	Genbank	region	1	29836	.	+	.	ID=RacCSB20637
RacCSB20637	Genbank	CDS	266	13444	.	+	.	ID=ORF1a
RacCSB20637	Genbank	CDS	13429	21516	.	+	.	ID=ORF1b
RacCSB20637	Genbank	CDS	21523	25320	.	+	.	ID=S
RacCSB20637	Genbank	CDS	25329	26156	.	+	.	ID=ORF3a
RacCSB20637	Genbank	CDS	25750	25818	.	+	.	ID=ORF3b
RacCSB20637	Genbank	CDS	26181	26408	.	+	.	ID=E
RacCSB20637	Genbank	CDS	26459	27124	.	+	.	ID=M
RacCSB20637	Genbank	CDS	27135	27320	.	+	.	ID=ORF6
RacCSB20637	Genbank	CDS	27327	27692	.	+	.	ID=ORF7a
RacCSB20637	Genbank	CDS	27689	27820	.	+	.	ID=ORF7b
RacCSB20637	Genbank	CDS	27827	28192	.	+	.	ID=ORF8
RacCSB20637	Genbank	CDS	28207	29466	.	+	.	ID=N
RacCSB20637	Genbank	CDS	28217	28510	.	+	.	ID=ORF9b
RacCSB20637	Genbank	CDS	29491	29607	.	+	.	ID=ORF10
